# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [8]:
# Your code goes here
import numpy as np
import pandas as pd

url_1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url_2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url_3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

dataset_1 = pd.read_csv(url_1)
display(dataset_1.head())

dataset_2 = pd.read_csv(url_2)
display(dataset_2.head())

dataset_3 = pd.read_csv(url_3)
display(dataset_3.head())

datasets = [dataset_1, dataset_2, dataset_3]

for dataset in datasets:
    dataset.columns = (
    dataset.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    )
    dataset.rename(columns={"st": "state"}, inplace=True)
    display(dataset.head())

dataset_combined = pd.concat(datasets, ignore_index=True)

display(dataset_combined.head())
dataset_combined.info()

def clean_data(dataset):
    dataset = dataset.copy()

    dataset = dataset.dropna(how="all")

    categorical_columns = [
        "gender", "state", "education", "policy_type", "vehicle_class"
    ]

    for column in categorical_columns:
        dataset[column] = dataset[column].str.strip()

    dataset["gender"] = dataset["gender"].replace({
        "Male": "M",
        "female": "F",
        "Femal": "F"
    })

    dataset["state"] = dataset["state"].replace({
        "Cali": "California",
        "WA": "Washington",
        "AZ": "Arizona"
    })

    dataset["education"] = dataset["education"].replace({
        "Bachelors": "Bachelor"
    })

    dataset["vehicle_class"] = dataset["vehicle_class"].replace({
        "Luxury SUV": "Luxury",
        "Luxury Car": "Luxury",
        "Sports Car": "Luxury"
    })

    dataset["customer_lifetime_value"] = pd.to_numeric(
        dataset["customer_lifetime_value"]
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    complaints = (
        dataset["number_of_open_complaints"]
        .astype("string")
        .str.strip()
    )

    complaints = complaints.where(
        ~complaints.str.contains("/", regex=False, na=False),
        complaints.str.split("/").str[1]
    )

    dataset["number_of_open_complaints"] = pd.to_numeric(
        complaints,
        errors="coerce"
    ).astype("Int64")

    dataset["gender"] = dataset["gender"].fillna("Unknown")

    dataset["customer_lifetime_value"] = (
        dataset["customer_lifetime_value"]
        .fillna(dataset["customer_lifetime_value"].median())
    )

    numeric_columns = [
        "customer_lifetime_value",
        "income",
        "monthly_premium_auto",
        "number_of_open_complaints",
        "total_claim_amount"
    ]

    for column in numeric_columns:
        dataset[column] = (
            pd.to_numeric(dataset[column], errors="coerce")
            .round()
            .astype("Int64")
        )

    dataset = dataset.drop_duplicates().reset_index(drop=True)

    return dataset

dataset_combined = clean_data(dataset_combined)

display(dataset_combined.head())
dataset_combined.info()

print("\nRemaining missing values:")
print(dataset_combined.isna().sum())

print("\nDuplicate rows:")
print(dataset_combined.duplicated().sum())

dataset_combined.to_csv("customer_data_cleaned.csv", index=False)

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Total Claim Amount,Policy Type,Vehicle Class
0,GS98873,Arizona,F,Bachelor,323912.47%,16061,88,1/0/00,633.6,Personal Auto,Four-Door Car
1,CW49887,California,F,Master,462680.11%,79487,114,1/0/00,547.2,Special Auto,SUV
2,MY31220,California,F,College,899704.02%,54230,112,1/0/00,537.6,Personal Auto,Two-Door Car
3,UH35128,Oregon,F,College,2580706.30%,71210,214,1/1/00,1027.2,Personal Auto,Luxury Car
4,WH52799,Arizona,F,College,380812.21%,94903,94,1/0/00,451.2,Corporate Auto,Two-Door Car


,Customer,State,Customer Lifetime Value,Education,Gender,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Total Claim Amount,Vehicle Class
0,SA25987,Washington,3479.137523,High School or Below,M,0,104,0,Personal Auto,499.200000,Two-Door Car
1,TB86706,Arizona,2502.637401,Master,M,0,66,0,Personal Auto,3.468912,Two-Door Car
2,ZL73902,Nevada,3265.156348,Bachelor,F,25820,82,0,Personal Auto,393.600000,Four-Door Car
3,KX23516,California,4455.843406,High School or Below,F,0,121,0,Personal Auto,699.615192,SUV
4,FN77294,California,7704.958480,High School or Below,M,30366,101,2,Personal Auto,484.800000,SUV


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,total_claim_amount,policy_type,vehicle_class
0,GS98873,Arizona,F,Bachelor,323912.47%,16061,88,1/0/00,633.6,Personal Auto,Four-Door Car
1,CW49887,California,F,Master,462680.11%,79487,114,1/0/00,547.2,Special Auto,SUV
2,MY31220,California,F,College,899704.02%,54230,112,1/0/00,537.6,Personal Auto,Two-Door Car
3,UH35128,Oregon,F,College,2580706.30%,71210,214,1/1/00,1027.2,Personal Auto,Luxury Car
4,WH52799,Arizona,F,College,380812.21%,94903,94,1/0/00,451.2,Corporate Auto,Two-Door Car


,customer,state,customer_lifetime_value,education,gender,income,monthly_premium_auto,number_of_open_complaints,policy_type,total_claim_amount,vehicle_class
0,SA25987,Washington,3479.137523,High School or Below,M,0,104,0,Personal Auto,499.200000,Two-Door Car
1,TB86706,Arizona,2502.637401,Master,M,0,66,0,Personal Auto,3.468912,Two-Door Car
2,ZL73902,Nevada,3265.156348,Bachelor,F,25820,82,0,Personal Auto,393.600000,Four-Door Car
3,KX23516,California,4455.843406,High School or Below,F,0,121,0,Personal Auto,699.615192,SUV
4,FN77294,California,7704.958480,High School or Below,M,30366,101,2,Personal Auto,484.800000,SUV


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12074 entries, 0 to 12073
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9137 non-null   object 
 1   state                      9137 non-null   object 
 2   gender                     9015 non-null   object 
 3   education                  9137 non-null   object 
 4   customer_lifetime_value    9130 non-null   object 
 5   income                     9137 non-null   float64
 6   monthly_premium_auto       9137 non-null   float64
 7   number_of_open_complaints  9137 non-null   object 
 8   policy_type                9137 non-null   object 
 9   vehicle_class              9137 non-null   object 
 10  total_claim_amount         9137 non-null   float64
dtypes: float64(3), object(8)
memory usage: 1.0+ MB


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Unknown,Master,7715,0,1000,0,Personal Auto,Four-Door Car,3
1,QZ44356,Arizona,F,Bachelor,697954,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,530
4,GA49547,Washington,M,High School or Below,536308,36357,68,0,Personal Auto,Four-Door Car,17


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9134 entries, 0 to 9133
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   customer                   9134 non-null   object
 1   state                      9134 non-null   object
 2   gender                     9134 non-null   object
 3   education                  9134 non-null   object
 4   customer_lifetime_value    9134 non-null   Int64 
 5   income                     9134 non-null   Int64 
 6   monthly_premium_auto       9134 non-null   Int64 
 7   number_of_open_complaints  9134 non-null   Int64 
 8   policy_type                9134 non-null   object
 9   vehicle_class              9134 non-null   object
 10  total_claim_amount         9134 non-null   Int64 
dtypes: Int64(5), object(6)
memory usage: 829.7+ KB

Remaining missing values:
customer                     0
state                        0
gender                       0
educat

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [11]:
# Your code goes here
# 1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
# Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

marketing_dataset = pd.read_csv(url)

display(marketing_dataset.head())
print(marketing_dataset.columns.tolist())

revenue_by_channel = marketing_dataset.pivot_table(
    index="sales_channel",
    values="monthly_premium_auto",
    aggfunc="sum"
).round(2)

display(revenue_by_channel)

# Agents generated the highest total monthly premiums, followed by branches and call centers. Web generated the lowest.

# 2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

average_clv = marketing_dataset.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

display(average_clv)

# Customers with highschool education or below had the highest average CLV for both genders, while Doctor customers had the lowest. Gender differences varied by education level.

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']


,monthly_premium_auto
sales_channel,
Agent,386335
Branch,280953
Call Center,197970
Web,151511


education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here